# AI Quiz Generator — Team MechMind

**Software project implementation** following the methodology slides:

1. **Input Acquisition** — text, PDF, or dataset context
2. **Text Preprocessing** — tokenization, stop-word removal, sentence segmentation
3. **Content Analysis** — keywords, entities (NLTK + optional spaCy)
4. **Question Generation** — rule-based MCQ, True/False, Short Answer
5. **Answer Extraction** — tied to source sentences
6. **Question Classification** — type field per question
7. **Quiz Structuring** — JSON-serializable output
8. **Output & Interface** — Flask web app (`app.py`)

## 0. Setup

In [31]:
%pip install -q nltk pypdf pandas flask spacy
# Optional: python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.


In [32]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from quiz_generator import (
    generate_quiz,
    quiz_to_json,
    load_dataset,
    sample_context,
    evaluate_metrics,
)
from quiz_generator.dataset import dataset_stats
from quiz_generator.pipeline import preprocess, analyze_content, acquire_input

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Tawhid\Downloads\AI lab project


## 1. Dataset overview (`final_dataset_projext.csv`)

In [33]:
rows = load_dataset()
stats = dataset_stats(rows)
print("Total rows:", stats["total_rows"])
print("Question types:", stats["question_types"])
print("Difficulties:", stats["difficulties"])
print("Top subjects:", stats["subjects"])

Total rows: 10570
Question types: {'mcq': 8341, 'short-answer': 2213, 'true-false': 16}
Difficulties: {'Easy': 10426, 'Medium': 136, 'Hard': 8}
Top subjects: {'History': 3371, 'General Knowledge': 2092, 'Physics & Engineering': 1542, 'Biology & Chemistry': 963, 'Computer Science': 879, 'Education': 560, 'Sports': 445, 'Politics & Government': 382, 'Economics': 336}


## 2. Step 1 — Input Acquisition

Load a **context** passage from the dataset (gold labels held out for comparison).

In [34]:
gold = sample_context(rows, seed=42)
text = str(gold["context"])

print("--- Gold labels (for evaluation only) ---")
print("Subject:", gold.get("subject"))
print("Type:", gold.get("question_type"))
print("Question:", gold["question"][:200])
print("Answer:", gold["answer"])
print("\nContext preview:", text[:400], "...")

--- Gold labels (for evaluation only) ---
Subject: Physics & Engineering
Type: mcq
Question: How might gravity effects be observed differently according to Newton?
Answer: at larger distances.

Context preview: Newton came to realize that the effects of gravity might be observed in different ways at larger distances. In particular, Newton determined that the acceleration of the Moon around the Earth could be ascribed to the same force of gravity if the acceleration due to gravity decreased as an inverse square law. Further, Newton realized that the acceleration due to gravity is proportional to the mass  ...


## 3. Step 2 — Text Preprocessing

In [35]:
clean_text, sentences, tokens_nostop = preprocess(text)
print("Sentences:", len(sentences))
print("Tokens (no stopwords):", len(tokens_nostop))
print("Sample sentence:", sentences[0][:150])

Sentences: 4
Tokens (no stopwords): 48
Sample sentence: Newton came to realize that the effects of gravity might be observed in different ways at larger distances.


## 4. Step 3 — Content Analysis

In [36]:
analysis = analyze_content(sentences, tokens_nostop)
print("Top keywords:", analysis["keywords"][:12])
print("Entities:", analysis["entities"][:8])

Top keywords: ['gravity', 'acceleration', 'newton', 'earth', 'due', 'mass', 'came', 'realize', 'effects', 'might', 'observed', 'different']
Entities: []


## 5. Steps 4–7 — Generate, classify, and structure quiz

In [37]:
quiz_items = generate_quiz(text, max_per_type=3, seed=123)
print(f"Generated {len(quiz_items)} questions\n")

for i, q in enumerate(quiz_items, 1):
    print(f"--- Q{i} [{q.qtype}] ---")
    print(q.question[:350])
    if q.options:
        for j, o in enumerate(q.options, 1):
            print(f"  {j}. {o}")
    print(f"Answer: {q.correct}\n")

Generated 4 questions

--- Q1 [MCQ] ---
Multiple choice — what fits the blank?
Newton came to realize that the effects of gravity might be observed in ________ ways at larger distances.
  1. different
  2. inverse
  3. realize
  4. could
Answer: different

--- Q2 [MCQ] ---
Multiple choice — what fits the blank?
In particular, Newton determined that the ________ of the Moon around the Earth could be ascribed to the same force of gravity if the acceleration due to gravity decreased as an inverse square law.
  1. inverse
  2. Moon
  3. Earth
  4. acceleration
Answer: acceleration

--- Q3 [MCQ] ---
Multiple choice — what fits the blank?
Further, Newton realized that the ________ due to gravity is proportional to the mass of the attracting body.
  1. square
  2. ascribed
  3. acceleration
  4. proportional
Answer: acceleration

--- Q4 [TrueFalse] ---
True or False: Combining these ideas gives a formula that relates the mass () and the radius () of the Earth to the gravitational acceleration

In [38]:
print("JSON export:")
print(quiz_to_json(quiz_items))

JSON export:
[
  {
    "type": "MCQ",
    "question": "Multiple choice — what fits the blank?\nNewton came to realize that the effects of gravity might be observed in ________ ways at larger distances.",
    "correct": "different",
    "source_sentence": "Newton came to realize that the effects of gravity might be observed in different ways at larger distances.",
    "options": [
      "different",
      "inverse",
      "realize",
      "could"
    ]
  },
  {
    "type": "MCQ",
    "question": "Multiple choice — what fits the blank?\nIn particular, Newton determined that the ________ of the Moon around the Earth could be ascribed to the same force of gravity if the acceleration due to gravity decreased as an inverse square law.",
    "correct": "acceleration",
    "source_sentence": "In particular, Newton determined that the acceleration of the Moon around the Earth could be ascribed to the same force of gravity if the acceleration due to gravity decreased as an inverse square law.",


## 6. Performance Metrics (from slides)

- **Relevance** — overlap with source text
- **Readability** — sentence clarity proxy
- **Accuracy proxy** — valid Q/A pairs generated

In [39]:
metrics = evaluate_metrics(quiz_items, text)
for k, v in metrics.items():
    print(f"{k}: {v}")

count: 4
valid_questions: 4
relevance_mean: 0.699
readability_mean: 1.0
accuracy_proxy: 1.0


## 7. Demo with custom sample text

In [40]:
SAMPLE = """
Photosynthesis is the process by which green plants use sunlight to synthesize nutrients from carbon dioxide and water.
Chlorophyll in the leaves absorbs light energy. Oxygen is released as a byproduct.
Mitochondria are organelles known as the powerhouse of the cell because they produce ATP through cellular respiration.
""".strip()

demo_items = generate_quiz(SAMPLE, max_per_type=2, seed=7)
print(evaluate_metrics(demo_items, SAMPLE))
print(quiz_to_json(demo_items))

{'count': 4, 'valid_questions': 4, 'relevance_mean': 0.592, 'readability_mean': 1.0, 'accuracy_proxy': 1.0}
[
  {
    "type": "MCQ",
    "question": "Multiple choice — what fits the blank?\n________ is the process by which green plants use sunlight to synthesize nutrients from carbon dioxide and water.",
    "correct": "Photosynthesis",
    "source_sentence": "Photosynthesis is the process by which green plants use sunlight to synthesize nutrients from carbon dioxide and water.",
    "options": [
      "produce",
      "green",
      "Photosynthesis",
      "carbon"
    ]
  },
  {
    "type": "MCQ",
    "question": "Multiple choice — what fits the blank?\n________ in the leaves absorbs light energy.",
    "correct": "Chlorophyll",
    "source_sentence": "Chlorophyll in the leaves absorbs light energy.",
    "options": [
      "absorbs",
      "Chlorophyll",
      "green",
      "leaves"
    ]
  },
  {
    "type": "TrueFalse",
    "question": "True or False: Oxygen is released as a bypr

## 8. Web Interface

Run in terminal from project folder:

```bash
python app.py
```

Open **http://127.0.0.1:5000** — paste text or upload a PDF to generate quizzes interactively.